Before we can move on to ablations and the two main experiments we need to do two things
1. Filter all 2100 buildings from false positives, there is no-buildings in the dataset. We want to claim 100% buildings 
2. Estimate $\delta$ where $\delta$ is the the distance at which no concrete distinction can be made (the label noise around the building edge) This consists of visual occlusions image quality etc and i estimate it at around 1m. However to correctly measure it, we first need to account for systemic offset in the data, the offset between footprints and buildings. If it is consistent we can correct the dataset by that amount, otherwise it is the grounded minimum for the value of $\delta$ 

In [4]:
# 1.

import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from rasterio.windows import Window

from thesis.data import naip
from thesis.run import REPO_ROOT, _geometry_to_crs, load_config

out_dir = Path("/Users/phillip/repositories/bachelor-thesis/data/dataset_v2")
imagery_config = load_config("imagery")
scene_cache = REPO_ROOT / imagery_config["scene_cache_dir"]

In [5]:
# kinda annoyed at all the hardcoding
selected = pd.read_parquet(out_dir / "selection" / "sources.parquet")
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")
boxes = gpd.read_parquet(out_dir / "inventory" / "boxes.parquet")
box_geometries = boxes.set_index("box_id").geometry.to_dict()

len(selected), len(pins)

(2100, 28893)

In [6]:
near = pins.loc[pins["bin_id"].isin([-1, 1])]
per_source_bins = near.groupby("source_id")["bin_id"].apply(set)
missing = selected.loc[~selected["source_id"].isin(per_source_bins.index)]
len(missing)  # if something missing big problem

0

In [7]:
one_per_source = (
    near.sort_values(["source_id", "bin_id"])
    .drop_duplicates(subset="source_id", keep="first")
    .merge(
        selected[["source_id", "box_id", "naip_item_id", "scene_window", "scene_transform"]],
        on="source_id",
    )
)
len(one_per_source)  # one pin per image, we wont render, just to filter

2100

In [8]:
# see run_santiy_render
import numpy as np

item_records: dict[str, dict[str, dict]] = {}


def get_item(box_id: str, naip_item_id: str) -> dict:
    if box_id not in item_records:
        box_wgs84 = _geometry_to_crs(box_geometries[box_id], "EPSG:5070", "EPSG:4326")
        item_records[box_id] = {
            str(item["id"]): item for item in naip.search_items_for_geometry(box_wgs84, imagery_config)
        }
    return item_records[box_id][str(naip_item_id)]


def raw_crop_for(row) -> np.ndarray:
    item = get_item(str(row["box_id"]), row["naip_item_id"])
    scene_window = Window(*[int(v) for v in row["scene_window"]])
    cache_key = naip.window_cache_key(str(row["naip_item_id"]), scene_window)
    tile = naip.load_cached(scene_cache, cache_key)
    if tile is None:
        tile = naip.read_item_window(
            item, scene_window, float(imagery_config["gsd_m"]), float(imagery_config["gsd_tolerance"])
        )
        naip.store_atomic(scene_cache, cache_key, tile.array, tile.metadata)
    col_off, row_off, width, height = [int(v) for v in row["crop_window"]]
    return tile.array[:, row_off : row_off + height, col_off : col_off + width]


from IPython.display import display


def display_array(crop: np.ndarray) -> None:
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(np.transpose(crop, (1, 2, 0)))
    ax.axis("off")
    display(fig)
    # to make it show before input blocks
    plt.close(fig)

In [9]:
log_path = out_dir / "gold" / "is_building.jsonl"
log_path.parent.mkdir(parents=True, exist_ok=True)

reviewed = set()
if log_path.exists():
    reviewed = {json.loads(line)["source_id"] for line in log_path.read_text().splitlines()}

remaining = one_per_source.loc[~one_per_source["source_id"].isin(reviewed)].reset_index(drop=True)
i = 0
len(remaining)

0

In [10]:
row = remaining.iloc[i]
display_array(raw_crop_for(row))
print(f"[{i + 1}/{len(remaining)}] source={row['source_id']} box={row['box_id']} bin={row['bin_id']}")

call = input("real building? (y/n, opt short reason): ")
with log_path.open("a") as f:
    f.write(
        json.dumps(
            {"source_id": row["source_id"], "is_building": call.strip().lower().startswith("y"), "reason": call.strip()}
        )
        + "\n"
    )
i += 1

IndexError: single positional indexer is out-of-bounds

In [11]:
gold_flags = pd.read_json(log_path, lines=True)
gold = selected.merge(gold_flags, on="source_id", how="inner")
gold = gold.loc[gold["is_building"]]

gold["assignment"].value_counts()

assignment
main        1850
ablation      92
Name: count, dtype: int64

8% false positives all filtered out, remainder is the ground turth for the benchmarks 
interesting: see detection rate of 'is building' for the LLM on the false positives 

Re-annotation (intra-annotator agreement). Blind: previous label not shown. 50 previous negatives + 50 previous positives, each stratified by ecoregion, shuffled.

In [12]:
R2_PER_CLASS = 50
R2_SEED = 0

first = (
    pd.read_json(log_path, lines=True)
    .merge(one_per_source, on="source_id")
    .merge(selected[["source_id", "ecoregion"]], on="source_id")
)


def stratified(frame, n):
    share = frame["ecoregion"].value_counts(normalize=True)
    return pd.concat(
        frame.loc[frame["ecoregion"] == eco].sample(min(k, (frame["ecoregion"] == eco).sum()), random_state=R2_SEED)
        for eco, k in (share * n).round().astype(int).items()
    )


r2_sample = (
    pd.concat(
        [
            stratified(first.loc[~first["is_building"]], R2_PER_CLASS),
            stratified(first.loc[first["is_building"]], R2_PER_CLASS),
        ]
    )
    .sample(frac=1, random_state=R2_SEED)
    .reset_index(drop=True)
)

r2_path = out_dir / "gold" / "is_building_r2.jsonl"
r2_done = set()
if r2_path.exists():
    r2_done = {json.loads(line)["source_id"] for line in r2_path.read_text().splitlines()}
r2_remaining = r2_sample.loc[~r2_sample["source_id"].isin(r2_done)].reset_index(drop=True)
j = 0
len(r2_sample), len(r2_remaining)

(97, 97)

In [110]:
row = r2_remaining.iloc[j]
display_array(raw_crop_for(row))
print(f"[{j + 1}/{len(r2_remaining)}] source={row['source_id']}")

call = input("real building? (y/n, opt short reason): ")
with r2_path.open("a") as f:
    f.write(
        json.dumps(
            {"source_id": row["source_id"], "is_building": call.strip().lower().startswith("y"), "reason": call.strip()}
        )
        + "\n"
    )
j += 1

IndexError: single positional indexer is out-of-bounds

In [113]:
r2 = pd.read_json(r2_path, lines=True).merge(
    first[["source_id", "is_building"]], on="source_id", suffixes=("_r2", "_r1")
)
{
    "n": len(r2),
    "agree_on_r1_positives": (r2.loc[r2["is_building_r1"], "is_building_r2"]).mean(),
    "agree_on_r1_negatives": (~r2.loc[~r2["is_building_r1"], "is_building_r2"]).mean(),
}

{'n': 97,
 'agree_on_r1_positives': np.float64(0.9795918367346939),
 'agree_on_r1_negatives': np.float64(0.9583333333333334)}

In [112]:
# Found during edge measurement (edge_displacement.ipynb, 2026-10-04): source is not a building.
# Flip its first-pass label in is_building.jsonl so every reader of the gold file excludes it. Idempotent.
LATE_REMOVALS = {"48962b6d-a782-404f-8dbf-f2803a5af2d5": "not building, found during edge measurement"}

records = [json.loads(line) for line in log_path.read_text().splitlines()]
for record in records:
    if record["source_id"] in LATE_REMOVALS:
        record.update({"is_building": False, "reason": LATE_REMOVALS[record["source_id"]]})
log_path.write_text("".join(json.dumps(record) + "\n" for record in records))
pd.read_json(log_path, lines=True).merge(selected, on="source_id").query("is_building")["assignment"].value_counts()

assignment
main        1849
ablation      92
Name: count, dtype: int64